In [1]:
import os
from libraries.lns.__main__ import run_alns
from libraries.lns.base_class import LNS_Base

from util.vrp_visualizer import plot_routes_from_distance_matrix

os.environ["BASE_PATH"] = "/home/nico/KIT_IMI/multibay-reshuffling"

lns = LNS_Base("cvrp")


/home/nico/.local/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [2]:
#################################################################
# CEOH Function

import copy

def ceoh_destroy(current, rng):
    new_state = copy.deepcopy(current)
    num_customers_to_remove = min(5, len(sum(new_state.routes, [])))
    removed_customers = []
    while len(removed_customers) < num_customers_to_remove:
        route_idx = rng.integers(len(new_state.routes))
        if new_state.routes[route_idx]:
            customer_idx = rng.choice(new_state.routes[route_idx])
            removed_customers.append(customer_idx)
            new_state.routes[route_idx].remove(customer_idx)
    new_state.unassigned = removed_customers
    return  new_state

import time as time_sleeping
def ceoh_repair(current, rng):
    while True:
        time_sleeping.sleep(1)
        print("Repair")
        pass


In [3]:
import os
from libraries.routefinder.util.data_loader import load_solutions, load_instances, load_npz_data
from libraries.routefinder.baselines.constants import ORTOOLS_SCALING_FACTOR
from libraries.routefinder.util.data_loader import load_instances
from libraries.routefinder.util.data_creation import create_dataset

def get_fitness(optimal, costs):
    costs /= ORTOOLS_SCALING_FACTOR
    return (costs - abs(optimal)) / abs(optimal)

def run_lns(lns_instance):
    lns_instance.run()
    return lns_instance.get_result().objective()


In [4]:
import traceback
from problems.function_runner import FunctionRunner

# ----------------------------
# Worker function
# ----------------------------
def run_lns_worker(data, index, stop_flag=None):
    """
    Worker for VRP problem with cooperative stopping.
    """
    try:
        lns = LNS_Base("cvrp")
        data["depots"] = 0
        lns.set_data(data)

        # Wrap destroy to accept any args, check stop_flag
        def destroy_wrapper(*args, **kwargs):
            if stop_flag is not None and stop_flag.is_set():
                raise Exception("Stop flag set, terminating destroy.")
            return ceoh_destroy(*args, **kwargs)

        lns.set_destroy(destroy_wrapper)
        lns.init_lns()

        result = lns.run()
        return (index, result)

    except Exception:
        traceback.print_exc()
        return (index, None)

# ----------------------------
# Main execution
# ----------------------------
NUMBER_OF_SAMPLES = 5
VRP_PROBLEM = "cvrp"

instances = load_instances(VRP_PROBLEM)
data_list = create_dataset(instances["val"]["50"])
sol = load_solutions(VRP_PROBLEM)["val"]["50"]
costs = load_npz_data(sol)["costs"]

tasks = []
results = [None] * NUMBER_OF_SAMPLES

# Create and start tasks
for i in range(NUMBER_OF_SAMPLES):
    task = FunctionRunner(run_lns_worker, i, data_list[i], i)
    tasks.append(task)
    task.start()

# Join with timeout
for t in tasks:
    finished = t.join(timeout=10)
    if not finished:
        print(f"Task {t.index} exceeded time limit. Stopping...")
        t.stop()  # signal inner threads to stop
        t.join(timeout=2)  # give a little time to clean up
    idx, res = t.index, t.result
    results[idx] = res

print("Results:", results)


/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The BoundedTensorSpec has been deprecated and will be removed in v0.8. Please use Bounded instead.
  warnings.warn(
/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The UnboundedDiscreteTensorSpec has been deprecated and will be removed in v0.8. Please use Unbounded instead.
  warnings.warn(
/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The CompositeSpec has been deprecated and will be removed in v0.8. Please use Composite instead.
  warnings.warn(
/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The UnboundedContinuousTensorSpec has been deprecated and will be removed in v0.8. Please use Unbounded instead.
  warnings.warn(


[LNS]: Loading data for the first time.
[LNS]: Loading data for the first time.
Generating initial solution with OR-Tools...
[LNS]: Loading data for the first time.
Generating initial solution with OR-Tools...
[LNS]: Loading data for the first time.
Generating initial solution with OR-Tools...
Generating initial solution with OR-Tools...
[LNS]: Loading data for the first time.
Generating initial solution with OR-Tools...
Solution: 1202489 
 with routes: [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [35, 7, 23, 10, 15, 49, 33, 22], [27, 36, 39, 11, 19, 18], [43, 12, 38, 5, 47, 1, 29, 21], [4, 13, 41, 46, 30, 16, 17, 14, 42, 20, 9], [25, 34, 2, 37, 45, 3, 40], [8, 26, 44, 31, 50, 32, 48], [6, 24, 28]]
Initial Solution:
Objective: 1202489   
Solution: 1312744 
 with routes: [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []

In [5]:

state = lns.get_result()

lns.plot_objective()
plot_routes_from_distance_matrix(state.data["distance_matrix"], state.routes, state.objective(), title="CEoH pop0 best result")

Final Solution:


AttributeError: 'NoneType' object has no attribute 'best_state'

In [ ]:
lns.get_objective_history()

In [ ]:

os.environ["BASE_PATH"] = "/home/nico/KIT_IMI/multibay-reshuffling"

sol = load_solutions("cvrp")["val"]["50"]

print(load_npz_data(sol))

costs = load_npz_data(sol)["costs"]
time = load_npz_data(sol)["time"]
actions = list(load_npz_data(sol)["actions"][0])

obj_optimal = -costs[0]

obj_ortools = 1138521 / ORTOOLS_SCALING_FACTOR
obj_lns_lns_raw = 1127833 / ORTOOLS_SCALING_FACTOR
obj_lns_ceoh = 1119990 / ORTOOLS_SCALING_FACTOR
obj_lns_ceoh_rerun = state.objective() / ORTOOLS_SCALING_FACTOR

fitness_ortools = get_fitness(obj_optimal, obj_ortools)
fitness_lns_raw = get_fitness(obj_optimal, obj_lns_lns_raw)
fitness_lns_ceoh = get_fitness(obj_optimal, obj_lns_ceoh)
fitness_lns_ceoh_rerun = get_fitness(obj_optimal, obj_lns_ceoh_rerun)

print(fitness_ortools)
print(fitness_lns_raw)
print(fitness_lns_ceoh)

raw_costs = [obj_ortools, obj_lns_lns_raw, obj_lns_ceoh, obj_lns_ceoh_rerun]
fitness_values = [fitness_ortools, fitness_lns_raw, fitness_lns_ceoh, fitness_lns_ceoh_rerun]



In [ ]:
import matplotlib.pyplot as plt

# --- Labels ---
labels = ['OR-Tools', 'LNS-Raw', 'CEOH-exp', 'CEOH-rerun']

# --- Plot 1: Fitness values ---
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(labels, fitness_values, color='skyblue')
plt.title("Fitness Scores")
plt.ylabel("Fitness")
plt.grid(axis='y')

# --- Plot 2: Raw Costs with Optimal Reference ---
plt.subplot(1, 2, 2)
plt.bar(labels, raw_costs, color='lightcoral')
plt.axhline(obj_optimal, color='green', linestyle='--', label=f"Optimal ({obj_optimal})")
plt.title("Raw Objective Values")
plt.ylabel("Cost")
plt.legend(loc='lower center')
plt.grid(axis='y')

plt.tight_layout()
plt.show()